<a href="https://colab.research.google.com/github/Sankalpa-Giri/Titanic-Survival-Prediction/blob/main/titanic_survival_prediction_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
ML Classification Pipeline — Titanic Survival Prediction
=========================================================
Author : Sankalpa Giri
GitHub : https://github.com/Sankalpa-Giri
Dataset: Kaggle Titanic (https://www.kaggle.com/c/titanic)

Pipeline covers:
  1. Exploratory Data Analysis (EDA)
  2. Feature Engineering & Preprocessing
  3. Model Training — Logistic Regression, Decision Tree, Random Forest
  4. Hyperparameter Tuning via GridSearchCV
  5. Evaluation — Accuracy, F1-score, ROC-AUC, Confusion Matrix
  6. Bias-Variance analysis via Learning Curves
"""

'\nML Classification Pipeline — Titanic Survival Prediction\n=========================================================\nAuthor : Sankalpa Giri\nGitHub : https://github.com/Sankalpa-Giri\nDataset: Kaggle Titanic (https://www.kaggle.com/c/titanic)\n\nPipeline covers:\n  1. Exploratory Data Analysis (EDA)\n  2. Feature Engineering & Preprocessing\n  3. Model Training — Logistic Regression, Decision Tree, Random Forest\n  4. Hyperparameter Tuning via GridSearchCV\n  5. Evaluation — Accuracy, F1-score, ROC-AUC, Confusion Matrix\n  6. Bias-Variance analysis via Learning Curves\n'

In [2]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import (train_test_split, cross_val_score, GridSearchCV, learning_curve, StratifiedKFold)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay)
from sklearn.pipeline import Pipeline, make_pipeline

warnings.filterwarnings("ignore")
np.random.seed(42)


In [4]:
# ──────────────────────────────────────────────
# 0. PATHS
# ──────────────────────────────────────────────
DATA_DIR   = "data"
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR,   exist_ok=True)


# ──────────────────────────────────────────────
# 1. LOAD DATA
# ──────────────────────────────────────────────

"""Load train.csv from data/ directory (Kaggle Titanic format)."""
path = os.path.join(DATA_DIR, "train.csv")
if not os.path.exists(path):
    raise FileNotFoundError(
        f"'{path}' not found.\n"
        "Download from https://www.kaggle.com/c/titanic/data "
        "and place train.csv inside the data/ folder."
    )
df = pd.read_csv(path)
print(df.head())
print(f"\n[load]  shape: {df.shape}")



   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  

[

In [5]:
# ──────────────────────────────────────────────
# 2. EDA
# ──────────────────────────────────────────────

"""Print summary stats and save key EDA plots."""
print("\n── EDA ────────────────────────────────")
print("\nMissing values:\n", df.isnull().sum())
print("\nData types:\n", df.dtypes)
print("\nTarget distribution:\n", df["Survived"].value_counts())

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle("EDA — Titanic Dataset", fontsize=14, fontweight="bold")

# Survival count
sns.countplot(x="Survived", data=df, ax=axes[0, 0], palette=["#c0392b", "#27ae60"])
axes[0, 0].set_title("Survival Count")
axes[0, 0].set_xticklabels(["Did not survive", "Survived"])

# Survival by Pclass
sns.countplot(x="Pclass", hue="Survived", data=df, ax=axes[0, 1], palette=["#c0392b", "#27ae60"])
axes[0, 1].set_title("Survival by Passenger Class")
axes[0, 1].legend(["Died", "Survived"])

# Age distribution
df["Age"].dropna().hist(bins=30, ax=axes[1, 0], color="#2980b9", edgecolor="white")
axes[1, 0].set_title("Age Distribution")
axes[1, 0].set_xlabel("Age")

# Correlation heatmap (numeric only)
num_cols = df.select_dtypes(include=np.number).drop(columns=["PassengerId"], errors="ignore")
sns.heatmap(num_cols.corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=axes[1, 1], linewidths=0.5)
axes[1, 1].set_title("Correlation Heatmap")

plt.tight_layout()
path = os.path.join(OUTPUT_DIR, "eda.png")
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.close()
print(f"[eda]   saved → {path}")



── EDA ────────────────────────────────

Missing values:
 PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Data types:
 PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

Target distribution:
 Survived
0    549
1    342
Name: count, dtype: int64
[eda]   saved → outputs/eda.png


In [6]:
# ──────────────────────────────────────────────
# 3. FEATURE ENGINEERING & PREPROCESSING
# ──────────────────────────────────────────────
"""
    Feature engineering decisions:
      - Title extracted from Name (Mr, Mrs, Miss, Master, Rare)
      - FamilySize = SibSp + Parch + 1
      - IsAlone flag
      - AgeBand after median imputation grouped by Title
      - FareBand after log1p transform
"""
df = df.copy()

# Title
df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
rare = [t for t, c in df["Title"].value_counts().items() if c < 10]
df["Title"] = df["Title"].replace(rare, "Rare")
df["Title"] = df["Title"].replace(
    {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"}
)

# Age imputation by median within Title group
df["Age"] = df.groupby("Title")["Age"].transform(
    lambda x: x.fillna(x.median())
)
df["Age"].fillna(df["Age"].median(), inplace=True)

# Fare imputation
df["Fare"].fillna(df["Fare"].median(), inplace=True)
df["Fare"] = np.log1p(df["Fare"])      # reduces right-skew

# Family features
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"]    = (df["FamilySize"] == 1).astype(int)

# Cabin: known / unknown
df["HasCabin"] = df["Cabin"].notna().astype(int)

# Embarked imputation
df["Embarked"].fillna(df["Embarked"].mode()[0], inplace=True)

# Encode categoricals
le = LabelEncoder()
for col in ["Sex", "Embarked", "Title"]:
    df[col] = le.fit_transform(df[col].astype(str))

feature_cols = [
    "Pclass", "Sex", "Age", "Fare",
    "FamilySize", "IsAlone", "HasCabin",
    "Embarked", "Title"
]
print(df[feature_cols + ["Survived"]])



     Pclass  Sex   Age      Fare  FamilySize  IsAlone  HasCabin  Embarked  \
0         3    1  22.0  2.110213           2        0         0         2   
1         1    0  38.0  4.280593           2        0         1         0   
2         3    0  26.0  2.188856           1        1         0         2   
3         1    0  35.0  3.990834           2        0         1         2   
4         3    1  35.0  2.202765           1        1         0         2   
..      ...  ...   ...       ...         ...      ...       ...       ...   
886       2    1  27.0  2.639057           1        1         0         2   
887       1    0  19.0  3.433987           1        1         1         2   
888       3    0  21.0  3.196630           4        0         0         2   
889       1    1  26.0  3.433987           1        1         1         0   
890       3    1  32.0  2.169054           1        1         0         1   

     Title  Survived  
0        2         0  
1        3         1  
2     

In [15]:
# ──────────────────────────────────────────────
# 4. TRAIN / VALIDATION SPLIT
# ──────────────────────────────────────────────
def split_data(df: pd.DataFrame):
    X = df[feature_cols]
    y = df["Survived"]
    return train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [8]:
# ──────────────────────────────────────────────
# 5. MODEL DEFINITIONS
# ──────────────────────────────────────────────
def build_models():
    """
    Three models wrapped in sklearn Pipelines (StandardScaler + classifier).
    Scaling affects Logistic Regression most but is applied uniformly
    for fair comparison.
    """
    models = {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    LogisticRegression(max_iter=1000, random_state=42))
        ]),
        "Decision Tree": Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    DecisionTreeClassifier(random_state=42))
        ]),
        "Random Forest": Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    RandomForestClassifier(random_state=42))
        ]),
    }
    return models


In [9]:
# ──────────────────────────────────────────────
# 6. CROSS-VALIDATION BASELINE
# ──────────────────────────────────────────────
def cross_validate_models(models: dict, X_train, y_train):
    """5-fold stratified CV on training set before any tuning."""
    print("\n── 5-Fold Cross-Validation (training set) ─")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results = {}
    for name, pipe in models.items():
        scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy")
        results[name] = scores
        print(f"  {name:<22} {scores.mean():.4f} ± {scores.std():.4f}")
    return results

In [10]:
# ──────────────────────────────────────────────
# 7. HYPERPARAMETER TUNING (Random Forest)
# ──────────────────────────────────────────────
def tune_random_forest(X_train, y_train):
    """GridSearchCV over key Random Forest hyperparameters."""
    print("\n── Hyperparameter Tuning — Random Forest ──")
    param_grid = {
        "clf__n_estimators":      [100, 200, 300],
        "clf__max_depth":         [None, 5, 10, 15],
        "clf__min_samples_split": [2, 5, 10],
        "clf__max_features":      ["sqrt", "log2"],
    }
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    RandomForestClassifier(random_state=42))
    ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    gs = GridSearchCV(pipeline, param_grid, cv=cv, scoring="accuracy", n_jobs=-1, verbose=0)
    gs.fit(X_train, y_train)
    print(f"  Best CV accuracy : {gs.best_score_:.4f}")
    print(f"  Best params      : {gs.best_params_}")
    return gs.best_estimator_

In [11]:
# ──────────────────────────────────────────────
# 8. EVALUATION
# ──────────────────────────────────────────────
def evaluate_model(model, X_test, y_test, name="Model"):
    """Compute and print accuracy, F1, ROC-AUC."""
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc     = accuracy_score(y_test, y_pred)
    f1      = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)

    print(f"\n── {name} — Test Metrics ───────────────")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  F1-score  : {f1:.4f}")
    print(f"  ROC-AUC   : {roc_auc:.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_test, y_pred, target_names=["Died", "Survived"]))
    return {"Accuracy": acc, "F1": f1, "ROC-AUC": roc_auc}

In [12]:
# ──────────────────────────────────────────────
# 9. PLOTS — CONFUSION MATRIX, ROC, LEARNING CURVE
# ──────────────────────────────────────────────
def plot_confusion_matrix(model, X_test, y_test):
    cm = confusion_matrix(y_test, model.predict(X_test))
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Died", "Survived"],
                yticklabels=["Died", "Survived"], ax=ax)
    ax.set_ylabel("Actual")
    ax.set_xlabel("Predicted")
    ax.set_title("Confusion Matrix — Tuned Random Forest")
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot]  saved → {path}")


def plot_roc_curve(models_eval: dict, X_test, y_test):
    """Overlay ROC curves for all evaluated models."""
    fig, ax = plt.subplots(figsize=(7, 5))
    for name, model in models_eval.items():
        RocCurveDisplay.from_estimator(model, X_test, y_test,
                                       ax=ax, name=name)
    ax.plot([0, 1], [0, 1], "k--", label="Random classifier")
    ax.set_title("ROC Curves — All Models")
    ax.legend(loc="lower right")
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "roc_curves.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot]  saved → {path}")


def plot_learning_curve(model, X_train, y_train):
    """
    Learning curve — diagnoses bias vs variance.
      - Large gap between train/val  → high variance (overfitting)
      - Both curves low and converged → high bias (underfitting)
    """
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_train, y_train,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring="accuracy", n_jobs=-1
    )
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(train_sizes, train_scores.mean(axis=1),
            "o-", color="#2980b9", label="Training accuracy")
    ax.fill_between(train_sizes,
                    train_scores.mean(axis=1) - train_scores.std(axis=1),
                    train_scores.mean(axis=1) + train_scores.std(axis=1),
                    alpha=0.15, color="#2980b9")
    ax.plot(train_sizes, val_scores.mean(axis=1),
            "o-", color="#e67e22", label="Validation accuracy")
    ax.fill_between(train_sizes,
                    val_scores.mean(axis=1) - val_scores.std(axis=1),
                    val_scores.mean(axis=1) + val_scores.std(axis=1),
                    alpha=0.15, color="#e67e22")
    ax.set_xlabel("Training set size")
    ax.set_ylabel("Accuracy")
    ax.set_title("Learning Curve — Tuned Random Forest\n"
                 "(bias-variance diagnostic)")
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "learning_curve.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot]  saved → {path}")


def plot_feature_importance(model, feature_names):
    """Bar chart of Random Forest feature importances."""
    rf    = model.named_steps["clf"]
    imps  = rf.feature_importances_
    order = np.argsort(imps)[::-1]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(range(len(imps)), imps[order],
           color="#8e44ad", edgecolor="white")
    ax.set_xticks(range(len(imps)))
    ax.set_xticklabels([feature_names[i] for i in order], rotation=40, ha="right")
    ax.set_ylabel("Importance")
    ax.set_title("Feature Importances — Tuned Random Forest")
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "feature_importance.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot]  saved → {path}")


def plot_model_comparison(results: dict):
    """Bar chart comparing all models on the three metrics."""
    metrics = ["Accuracy", "F1", "ROC-AUC"]
    model_names = list(results.keys())
    x = np.arange(len(metrics))
    width = 0.2
    colors = ["#2980b9", "#e67e22", "#27ae60", "#8e44ad"]

    fig, ax = plt.subplots(figsize=(9, 5))
    for i, (name, vals) in enumerate(results.items()):
        ax.bar(x + i * width,
               [vals[m] for m in metrics],
               width, label=name, color=colors[i], edgecolor="white")
    ax.set_xticks(x + width * (len(model_names) - 1) / 2)
    ax.set_xticklabels(metrics)
    ax.set_ylim(0.5, 1.0)
    ax.set_ylabel("Score")
    ax.set_title("Model Comparison — Test Set Metrics")
    ax.legend()
    ax.grid(True, axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "model_comparison.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot]  saved → {path}")

In [13]:
# ──────────────────────────────────────────────
# 10. MAIN
# ──────────────────────────────────────────────
def main():
    # Split
    X_train, X_test, y_train, y_test = split_data(df)
    print(f"\n[split] train={len(X_train)}  test={len(X_test)}")

    # Baseline models + CV
    models = build_models()
    cross_validate_models(models, X_train, y_train)

    # Fit baselines on full training set for evaluation
    for m in models.values():
        m.fit(X_train, y_train)

    # Tune Random Forest
    best_rf = tune_random_forest(X_train, y_train)

    # Evaluate all
    all_results = {}
    for name, model in models.items():
        all_results[name] = evaluate_model(model, X_test, y_test, name)
    all_results["RF (Tuned)"] = evaluate_model(
        best_rf, X_test, y_test, "RF (Tuned)"
    )

    # Plots
    feature_names = list(df.drop("Survived", axis=1).columns)
    plot_confusion_matrix(best_rf, X_test, y_test)
    plot_roc_curve(
        {"Logistic Regression": models["Logistic Regression"],
         "Decision Tree":       models["Decision Tree"],
         "Random Forest":       models["Random Forest"],
         "RF (Tuned)":          best_rf},
        X_test, y_test
    )
    plot_learning_curve(best_rf, X_train, y_train)
    plot_feature_importance(best_rf, feature_names)
    plot_model_comparison(all_results)

    print("\n── Final Results ───────────────────────")
    for name, vals in all_results.items():
        print(f"  {name:<22} Acc={vals['Accuracy']:.4f}  "
              f"F1={vals['F1']:.4f}  AUC={vals['ROC-AUC']:.4f}")
    print(f"\n[done]  All plots saved to outputs/")

In [16]:
if __name__ == "__main__":
    main()


[split] train=712  test=179

── 5-Fold Cross-Validation (training set) ─
  Logistic Regression    0.7992 ± 0.0223
  Decision Tree          0.7781 ± 0.0091
  Random Forest          0.8160 ± 0.0141

── Hyperparameter Tuning — Random Forest ──
  Best CV accuracy : 0.8315
  Best params      : {'clf__max_depth': 10, 'clf__max_features': 'sqrt', 'clf__min_samples_split': 2, 'clf__n_estimators': 100}

── Logistic Regression — Test Metrics ───────────────
  Accuracy  : 0.8101
  F1-score  : 0.7500
  ROC-AUC   : 0.8495

  Classification Report:

              precision    recall  f1-score   support

        Died       0.84      0.85      0.85       110
    Survived       0.76      0.74      0.75        69

    accuracy                           0.81       179
   macro avg       0.80      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179


── Decision Tree — Test Metrics ───────────────
  Accuracy  : 0.7486
  F1-score  : 0.6763
  ROC-AUC   : 0.7246

  Classification 